# Vector Search

## 2.2 Embeddings

In [ ]:
import time
from tqdm.auto import tqdm
import numpy as np
from numpy.typing import NDArray
from sentence_transformers import SentenceTransformer

# Some tyoedefs
Embedding = NDArray[np.float32]

In [39]:
q1: str = "I just discovered the course, can I still join?"
q2: str = "I just found out about the program, can I still enroll?"

In [40]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
v1 = model.encode(q1)

In [42]:
v1.shape

(384,)

Encoding the input query:

In [43]:
q1: str = 'Can I still join the course after the start date?'
v1: Embedding = model.encode(q1)  # Query vector

Encoding a "document":

In [44]:
d: str = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv: Embedding = model.encode(d)  # Document vector

Computing similarity (here: cos-simularity with normalized vectors) of encoded values of query and document:

In [45]:
sim: float = v1 @ dv  # Dot product of query and document vectors
print(f"Cosine similarity: {sim:.4f}")

Cosine similarity: 0.3233


A different unrelated query:

In [46]:
q2: str = 'How to install Docker on Windows?'
v2: Embedding = model.encode(q2)  # Query vector

Computing similarity of encoded values of unrelated query and document:

In [47]:
sim_2: float = v2 @ dv  # Dot product of query and document vectors
print(f"Cosine similarity: {sim_2:.4f}")

Cosine similarity: 0.0197


## 2.3 Embedding Our Dataset

Loading the FAQ data from all DataTalksClub courses:

In [48]:
from ingest import load_faq_data

documents: list[dict[str, str]] = load_faq_data()

In [49]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

Generating a list question-answer texts:

In [50]:
texts: list[str] = [doc["question"] + doc["answer"] for doc in documents]

In [51]:
print(len(texts))

1350


Chunking the documents into chunks of 50, which are then embedded:

In [ ]:
def embed_documents(texts: list[str], chunk_size: int = 50) -> list[Embedding]:
    """Embed a list of texts using SentenceTransformer model.

    Parameters
    ----------
    texts: list[str]
        The QA-texts to embed.

    Returns
    -------
    list[Embedding]
        The embedded texts.
    """
    vectors: list[Embedding] = []

    for i in tqdm(range(0, len(texts), chunk_size)):
        batch: list[str] = texts[i : i + chunk_size]
        batch_vectors: list[Embedding] = model.encode(batch)
        vectors.extend(batch_vectors)
    return vectors

vectors: list[Embedding] = embed_documents(texts, chunk_size=50)

  0%|          | 0/27 [00:00<?, ?it/s]

In [54]:
len(vectors)

1350

## 2.4 Vector Search

Checking the similarity of the query-embedding $\mathbf{v}_1$ with a random embedding vector from the dataset:

In [58]:
v1 @ vectors[10]

np.float32(0.33153278)

In [93]:
vv_t0 = time.time() # Measure time taken for vector-vector multiplication
best_match_idx: int  = np.argmax([v1 @ v for v in vectors])
vv_t1 = time.time()

print(f"Best match index: {best_match_idx}")
print(f"Best match similarity: {v1 @ vectors[best_match_idx]:.4f}")
print(f"Time taken: {vv_t1 - vv_t0:.4f} seconds")

Best match index: 2
Best match similarity: 0.7629
Time taken: 0.0032 seconds


Even better / faster than vector-vector multiplication is matrix-vector multiplication. For this all embeddings get stacked to a matrix and then multiplied with the query-embedding:

In [94]:
mv_t0 = time.time() # Measure time taken for matrix-vector multiplication
X = np.array(vectors)
mv_t1 = time.time()
print(X.shape)  # shape: (num_embeddings, embedding_dim)
print(f"Time taken: {mv_t1 - mv_t0:.4f} seconds")

(1350, 384)
Time taken: 0.0015 seconds


In [66]:
scores = X @ v1
print(scores.shape)  # shape: (num_embeddings,)

(1350,)


Get the highest similarity score from the similarity computations:

In [69]:
idx = np.argmax(scores)
int(idx), float(scores[idx])

(2, 0.7629410028457642)

In [70]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [82]:
top_5_idx = np.argsort(-scores)[:5]  # same as np.argsort(scores)[-5:] -> negative scores
print("top 5 indices:", top_5_idx)
top_5_scores = scores[top_5_idx]
print("top 5 scores:", top_5_scores)

for idx in top_5_idx:
    print(scores[idx])
    print(documents[idx])
    print()



top 5 indices: [  2 625 907 538   7]
top 5 scores: [0.762941   0.7579373  0.7192131  0.6536312  0.56009996]
0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579373
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8

## 2.5 Vector Search with `minsearch`

In [95]:
from minsearch import VectorSearch

In [96]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [102]:
vindex.search(
    v1,
    num_results=5,
    filter_dict={"course": "llm-zoomcamp"},
)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

## 2.6 RAG with Vector Search

In [103]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Get openai llm for use in RAG
openai_client = OpenAI()

In [106]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)  # Get index generated from minsearch

In [109]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [111]:
query: str = "I just found out about the program, can I still enroll?"
assistant.rag(query)

'Yes, you can still enroll. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [113]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query: str, num_results: int = 5) -> str:
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [115]:
query: str = "I just found out about the program, can I still enroll?"
vector_assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'